# 🗄️ SQL (Structured Query Language) — Complete Masterclass
> **Course Track:** Prime AI/ML Batch  
> **Source:** Apna College Curriculum (`SQL_Notes_by_Apna_College.pdf`)  
> **Topic:** DDL, DQL, DML, TCL, Joins, Group By, Having, Set Operations & Subqueries  
> **Language / Notes Mode:** Easy Natural Hinglish  

---

### 📌 Database Aur SQL Kya Hai?
- **Database:** Data ka ek organized, structured electronic collection jo Excel ya CSV files ke mukable million times fast, safe aur multi-user access support karta hai.
- **Relational Database (RDBMS):** Tables (rows & columns) ke roop me data store karta hai aur primary/foreign keys ke through relationships maintain karta hai.
- **SQL (Structured Query Language):** Relational database se baat karne ki standard international language hai (CRUD operations: Create, Read, Update, Delete).
- **SQL vs MySQL:** SQL ek language hai, jabki MySQL ek Relational Database Management System (Software Engine) hai jo SQL language ko samajhta hai.

> 💡 **Notebook Runner Note:** Is notebook me hum Python ke built-in `sqlite3` engine aur `pandas` ka use karenge taaki aap directly saari SQL queries ko bina kisi external MySQL server setup ke live execute kar sakein!


## 🏗️ Operation 1: DDL (Data Definition Language) — Tables & Constraints

### 💡 Kyu Zaroorat Hai?
Data store karne se pehle table ka blueprint (schema) define karna padta hai: columns ke naam, unke data types aur rules (constraints).

### 🔍 Key Constraints:
- `PRIMARY KEY`: Unique identifier (No duplicate, No NULL).
- `NOT NULL`: Column empty nahi chhod sakte.
- `UNIQUE`: Sare values alag-alag honge.
- `DEFAULT`: Agar value na di jaye toh default value fill ho jayegi.
- `CHECK`: Condition validate karta hai (e.g. `age >= 18`).


In [ ]:
import sqlite3
import pandas as pd

# In-memory SQL database create karna
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

def run_query(sql):
    """Helper function jo SQL query execute karke clean Pandas table return karta hai"""
    return pd.read_sql_query(sql, conn)

# 1. DDL: Create Tables with Constraints
cursor.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    age INTEGER CHECK(age >= 18),
    city VARCHAR(30) DEFAULT 'Bhopal',
    cgpa REAL
);
""")
conn.commit()

print("✅ Table 'students' successfully created with constraints.")


## 📝 Operation 2: DML (Data Manipulation Language) — INSERT, UPDATE, DELETE

### 💡 Kyu Zaroorat Hai?
Table structure banne ke baad usme records daalna, galat data ko update karna aur purana data delete karna DML kehlata hai.

### 🔍 DML Commands:
- `INSERT INTO table_name (columns) VALUES (...)`: Naye records add karna.
- `UPDATE table_name SET col = val WHERE condition`: Existing records modify karna.
- `DELETE FROM table_name WHERE condition`: Specific rows delete karna. (⚠️ `WHERE` lagana mat bhulna, warna poori table khali ho jayegi!).


In [ ]:
# 1. INSERT Records
cursor.executemany("""
INSERT INTO students (student_id, name, age, city, cgpa)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, 'Aarav Sharma', 21, 'Delhi', 8.9),
    (2, 'Isha Verma', 22, 'Mumbai', 9.4),
    (3, 'Rohan Mehta', 20, 'Bangalore', 7.5),
    (4, 'Sneha Patel', 23, 'Delhi', 9.1),
    (5, 'Vikram Singh', 21, 'Bhopal', 8.2)
])
conn.commit()

# 2. UPDATE Record: Rohan ka CGPA update karna
cursor.execute("""
UPDATE students 
SET cgpa = 8.0 
WHERE student_id = 3;
""")
conn.commit()

# 3. SELECT karke verify karna
run_query("SELECT * FROM students;")


## 🔍 Operation 3: DQL (Data Query Language) — SELECT & Conditional Filtering

### 💡 Kyu Zaroorat Hai?
Database se specific columns aur specific conditions satisfy karne wale records nikalne ke liye `SELECT ... FROM ... WHERE ...` use hota hai.

### 🔍 Logical Operators:
- `AND`: Jab dono conditions True hon.
- `OR`: Jab dono me se koi ek condition True ho.
- `NOT`: Condition ko reverse / negate karna.
- Comparisons: `=`, `!=` (ya `<>`), `>`, `<`, `>=`, `<=`.


In [ ]:
# 1. Specific Columns Select Karna
print("--- Specific Columns (name, cgpa) ---")
display(run_query("SELECT name, cgpa FROM students;"))

# 2. WHERE with AND: Age >= 21 AND CGPA > 8.5
print("\n--- Students with Age >= 21 AND CGPA > 8.5 ---")
display(run_query("""
SELECT name, age, city, cgpa 
FROM students 
WHERE age >= 21 AND cgpa > 8.5;
"""))

# 3. WHERE with OR: City Delhi OR City Bhopal
print("\n--- Students from Delhi OR Bhopal ---")
display(run_query("""
SELECT name, city, cgpa 
FROM students 
WHERE city = 'Delhi' OR city = 'Bhopal';
"""))


## 🎯 Operation 4: Pattern Matching (LIKE), Ranges (BETWEEN) & Lists (IN)

### 💡 Kyu Zaroorat Hai?
- `LIKE`: Text search ke liye wildcards:
  - `%`: Zero ya multiple characters (e.g. `'A%'` = starts with 'A', `'%sharma'` = ends with 'sharma').
  - `_`: Exactly one single character.
- `BETWEEN a AND b`: Numeric ya date range me search karna (inclusive).
- `IN (val1, val2, ...)`: Multiple OR conditions ki jagah clean list match karna.
- `DISTINCT`: Duplicate values ko result se remove karna.


In [ ]:
# 1. LIKE: Names starting with 'A' or 'S'
print("--- Names starting with 'A' (LIKE 'A%') ---")
display(run_query("SELECT * FROM students WHERE name LIKE 'A%';"))

# 2. BETWEEN: CGPA between 8.0 and 9.2
print("\n--- CGPA Between 8.0 and 9.2 ---")
display(run_query("SELECT name, cgpa FROM students WHERE cgpa BETWEEN 8.0 AND 9.2;"))

# 3. IN: City in ('Mumbai', 'Bangalore')
print("\n--- Students in Metro Cities (IN clause) ---")
display(run_query("SELECT name, city FROM students WHERE city IN ('Mumbai', 'Bangalore');"))

# 4. DISTINCT: Unique cities
print("\n--- Unique Cities (DISTINCT) ---")
display(run_query("SELECT DISTINCT city FROM students;"))


## 🔀 Operation 5: Sorting & Limiting Records (ORDER BY & LIMIT)

### 💡 Kyu Zaroorat Hai?
Top rankers dekhne ke liye highest-to-lowest CGPA sort karna (`DESC`) ya alphabetically sort karna (`ASC`).

### 🔍 Syntax:
- `ORDER BY col1 [ASC|DESC], col2 [ASC|DESC]`: Sorting sequence.
- `LIMIT n`: Sirf top n rows retrieve karna.


In [ ]:
# Top 3 students with highest CGPA
print("--- Top 3 Rankers (ORDER BY cgpa DESC LIMIT 3) ---")
display(run_query("""
SELECT name, cgpa, city 
FROM students 
ORDER BY cgpa DESC 
LIMIT 3;
"""))

# Multi-column Sort: City ASC, then CGPA DESC
print("\n--- Sorted by City then CGPA ---")
display(run_query("""
SELECT city, name, cgpa 
FROM students 
ORDER BY city ASC, cgpa DESC;
"""))


## 📊 Operation 6: Aggregate Functions, GROUP BY & The HAVING Clause

### 💡 Kyu Zaroorat Hai? (Frequently Asked Interview Question!)
- **Aggregate Functions:** `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()`.
- **`GROUP BY`:** Data ko categories (e.g. Cities, Departments) me divide karta hai.
- **`HAVING` vs `WHERE` Difference:**
  - `WHERE` clause rows par filter lagata hai **grouping hone se pehle**.
  - `HAVING` clause aggregated values par filter lagata hai **grouping hone ke baad** (jaise `HAVING COUNT(*) > 1` ya `HAVING AVG(cgpa) > 8.5`).


In [ ]:
# 1. Overall Aggregates
print("--- Overall Student Statistics ---")
display(run_query("""
SELECT 
    COUNT(*) AS total_students,
    ROUND(AVG(cgpa), 2) AS average_cgpa,
    MIN(cgpa) AS lowest_cgpa,
    MAX(cgpa) AS highest_cgpa
FROM students;
"""))

# 2. GROUP BY City: Har city ke students count aur average CGPA
print("\n--- City-wise Aggregations ---")
display(run_query("""
SELECT 
    city,
    COUNT(*) AS student_count,
    ROUND(AVG(cgpa), 2) AS avg_city_cgpa
FROM students
GROUP BY city;
"""))

# 3. GROUP BY with HAVING: Sirf wo cities jaha 1 se zyada students hain
print("\n--- Cities having more than 1 student (HAVING COUNT(*) > 1) ---")
display(run_query("""
SELECT 
    city,
    COUNT(*) AS student_count
FROM students
GROUP BY city
HAVING COUNT(*) > 1;
"""))


## 🔗 Operation 7: Relational Joins (INNER JOIN, LEFT JOIN, CROSS JOIN)

### 💡 Kyu Zaroorat Hai?
Real databases normalize hote hain — students ka data alag table me hota hai aur unke enrolled courses ka data alag table me. Joins se dono tables ko common foreign key se joda jata hai.

### 🔍 Types of Joins:
1. **INNER JOIN:** Sirf wahi rows jo dono tables me match karti hon.
2. **LEFT JOIN:** Left table ki saari rows aayengi; right me match na hone par `NULL` aayega.
3. **CROSS JOIN (Cartesian Product):** Table 1 ki har row Table 2 ki har row ke sath pair hoti hai ($M \times N$ rows).


In [ ]:
# Courses aur Enrollments tables create karna
cursor.execute("""
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name VARCHAR(50),
    credits INTEGER
);
""")

cursor.execute("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER
);
""")

cursor.executemany("INSERT INTO courses VALUES (?, ?, ?)", [
    (101, 'Machine Learning', 4),
    (102, 'Deep Learning', 4),
    (103, 'Data Structures', 3),
    (104, 'Web Development', 3)
])

cursor.executemany("INSERT INTO enrollments VALUES (?, ?, ?)", [
    (1, 1, 101),
    (2, 1, 103),
    (3, 2, 101),
    (4, 2, 102),
    (5, 4, 101),
    (6, 99, 104)  # Notice student 99 doesn't exist in students table
])
conn.commit()

# 1. INNER JOIN: Student Name + Course Name
print("--- INNER JOIN: Active Student Course Enrollments ---")
display(run_query("""
SELECT 
    s.name AS Student_Name,
    c.course_name AS Course,
    c.credits AS Credits
FROM enrollments e
INNER JOIN students s ON e.student_id = s.student_id
INNER JOIN courses c ON e.course_id = c.course_id;
"""))

# 2. LEFT JOIN: All Students (even if they didn't enroll in any course)
print("\n--- LEFT JOIN: All Students & Their Courses ---")
display(run_query("""
SELECT 
    s.name AS Student,
    c.course_name AS Enrolled_Course
FROM students s
LEFT JOIN enrollments e ON s.student_id = e.student_id
LEFT JOIN courses c ON e.course_id = c.course_id;
"""))


## 🪞 Operation 8: Self Join & Hierarchical Relationships

### 💡 Kyu Zaroorat Hai?
Jab ek hi table me hierarchy ya parent-child relation ho (jaise Employee aur unke Manager dono same `employees` table me rehte hain), tab hum **Self Join** use karte hain.


In [ ]:
# Manager Hierarchy Table
cursor.execute("""
CREATE TABLE staff (
    staff_id INTEGER PRIMARY KEY,
    name VARCHAR(30),
    manager_id INTEGER
);
""")

cursor.executemany("INSERT INTO staff VALUES (?, ?, ?)", [
    (1, 'Alice Johnson', 3),
    (2, 'Bob Rivera', 3),
    (3, 'Carol Smith (CEO)', NULL),
    (4, 'David Kim', 1)
])
conn.commit()

# Self Join: Staff member paired with their Manager's name
display(run_query("""
SELECT 
    e.name AS Employee_Name,
    COALESCE(m.name, 'Top Level Executive') AS Reports_To
FROM staff e
LEFT JOIN staff m ON e.manager_id = m.staff_id;
"""))


## 🧅 Operation 9: Subqueries (Nested / Inner Queries)

### 💡 Kyu Zaroorat Hai?
Jab query ka criteria kisi doosri query ke result par depend kare — jaise: "Un sabhi students ki list dikhao jinka CGPA pure batch ke average CGPA se zyada ho".

### 🔍 Kaise Kaam Karta Hai?
- Pehle inner query execute hoti hai (`(SELECT AVG(cgpa) FROM students)`).
- Fir outer query us value ko filter condition ke roop me use karti hai.


In [ ]:
# Subquery: Students with CGPA > Average CGPA
print("--- Students with CGPA Above Class Average ---")
display(run_query("""
SELECT name, cgpa, city
FROM students
WHERE cgpa > (SELECT AVG(cgpa) FROM students);
"""))

# Subquery with IN: Students who are enrolled in 'Machine Learning'
print("\n--- Students enrolled in Machine Learning (Using Subquery with IN) ---")
display(run_query("""
SELECT name, city 
FROM students 
WHERE student_id IN (
    SELECT student_id 
    FROM enrollments 
    WHERE course_id = (SELECT course_id FROM courses WHERE course_name = 'Machine Learning')
);
"""))


## 🛡️ Operation 10: TCL (Transaction Control Language) — COMMIT & ROLLBACK

### 💡 Kyu Zaroorat Hai?
Bank transactions me agar account A se paise deduct ho jayein aur server crash hone ki wajah se account B me credit na hon, toh system inconsistency create ho jati hai.
- **`COMMIT`:** Transaction ko permanently save karta hai.
- **`ROLLBACK`:** Kisi error par poori transaction ko undo karke wapis safe state par le aata hai.


In [ ]:
# Bank Accounts simulation table
cursor.execute("""
CREATE TABLE bank_accounts (
    acc_no INTEGER PRIMARY KEY,
    holder VARCHAR(30),
    balance REAL
);
""")

cursor.executemany("INSERT INTO bank_accounts VALUES (?, ?, ?)", [
    (101, 'Aman', 15000.0),
    (102, 'Priya', 8000.0)
])
conn.commit()

print("Initial Balances:")
display(run_query("SELECT * FROM bank_accounts;"))

# Transaction Example: Aman transfers 3000 to Priya
try:
    cursor.execute("UPDATE bank_accounts SET balance = balance - 3000 WHERE acc_no = 101;")
    cursor.execute("UPDATE bank_accounts SET balance = balance + 3000 WHERE acc_no = 102;")
    conn.commit()
    print("\n✅ Transaction Committed Successfully!")
except Exception as e:
    conn.rollback()
    print("\n❌ Error occurred, Transaction Rolled Back:", e)

print("\nFinal Balances after Transaction:")
display(run_query("SELECT * FROM bank_accounts;"))
